# Phase 1: Colab Environment Setup

Set up the full SoARM + LIBERO + OpenVLA-OFT stack on a fresh Colab A100 runtime.

## Requirements

- [ ] ENV-01: All dependencies install in correct order without pip resolver conflicts
- [ ] ENV-02: EGL headless rendering configured — OffScreenRenderEnv produces non-black LIBERO frames
- [ ] ENV-03: OpenVLA-OFT loads on GPU (A100 bf16) and returns a valid 7-D action tensor

## Usage

**BLOCK A** (this block): Run cells 0-9 top to bottom, then restart the runtime.

**BLOCK B** (Plan 02): After restart, run verification cells for ENV-01 / ENV-02 / ENV-03.

> Note: Block A must complete fully before restarting. Do not run Block B cells before restart.

In [1]:
import os

# ── USER CONFIGURATION ──────────────────────────────────────────────────────
# Set REPO_ROOT to the path where you cloned SoARM-Research.
# If using Google Drive: "/content/drive/MyDrive/SoARM-Research"
# If using git clone directly to Colab: "/content/SoARM-Research"
REPO_ROOT = "/content/drive/MyDrive/SoARM-Research"
# ────────────────────────────────────────────────────────────────────────────

# Derived path constants (do not edit these)
LIBERO_ROOT = f"{REPO_ROOT}/LIBERO/libero/libero"
LIBERO_PKG  = f"{REPO_ROOT}/LIBERO"   # path to setup.py directory
OUT_DIR     = f"{REPO_ROOT}/LIBERO/notebooks/outputs"
BDDL_FILE   = (
    f"{LIBERO_ROOT}/bddl_files/libero_spatial/"
    "pick_up_the_black_bowl_from_table_center_and_place_it_on_the_plate.bddl"
)

# Create outputs directory so Block B render check can save there
os.makedirs(OUT_DIR, exist_ok=True)

print(f"REPO_ROOT   = {REPO_ROOT}")
print(f"LIBERO_ROOT = {LIBERO_ROOT}")
print(f"LIBERO_PKG  = {LIBERO_PKG}")
print(f"OUT_DIR     = {OUT_DIR}")
print(f"BDDL_FILE   = {BDDL_FILE}")
print(f"Saved → {OUT_DIR}  (outputs directory ready)")

REPO_ROOT   = /content/drive/MyDrive/SoARM-Research
LIBERO_ROOT = /content/drive/MyDrive/SoARM-Research/LIBERO/libero/libero
LIBERO_PKG  = /content/drive/MyDrive/SoARM-Research/LIBERO
OUT_DIR     = /content/drive/MyDrive/SoARM-Research/LIBERO/notebooks/outputs
BDDL_FILE   = /content/drive/MyDrive/SoARM-Research/LIBERO/libero/libero/bddl_files/libero_spatial/pick_up_the_black_bowl_from_table_center_and_place_it_on_the_plate.bddl
Saved → /content/drive/MyDrive/SoARM-Research/LIBERO/notebooks/outputs  (outputs directory ready)


In [2]:
# GPU assertion — D-06
# Check GPU availability and warn loudly if not A100.
# OpenVLA-OFT in bf16 requires ~16 GB VRAM; A100 (40 GB) is the target.
import torch

assert torch.cuda.is_available(), (
    "No GPU available. Go to Runtime > Change runtime type > Hardware accelerator > GPU."
)

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9

print(f"GPU:  {gpu_name}")
print(f"VRAM: {vram_gb:.1f} GB")

if "A100" not in gpu_name:
    print()
    print("WARNING: Expected A100, got", gpu_name)
    print("WARNING: OpenVLA-OFT in bf16 requires ~16 GB+ VRAM.")
    print("WARNING: ENV-03 will OOM on T4 (15 GB). Restart with an A100 runtime.")
    print("WARNING: You may still proceed for install-only testing on T4.")
else:
    print("A100 confirmed. Proceeding.")

GPU:  Tesla T4
VRAM: 15.6 GB



---

## BLOCK A: Install

Run all cells in this block **top to bottom**, then restart the runtime.

**Ordering is critical** — do not reorder or skip cells:

1. EGL system packages (apt) must install before pip mujoco
2. PyTorch must install before flash-attn (flash-attn compiles against torch CUDA headers)
3. The custom transformers fork must install last (prevents pip downgrade to PyPI version)

---

In [3]:
# Step 1 of 6 — EGL system packages
# Must run BEFORE pip mujoco install.
# These C libraries must exist when the mujoco Python extension builds.
# apt-get update refreshes repo index — prevents 404 for stale package URLs (e.g. libosmesa6).
!apt-get update -qq
!apt-get install -y -q --fix-missing \
    libglfw3 \
    libglew-dev \
    libosmesa6-dev \
    libgles2 \
    libglvnd0 \
    libegl-dev \
    libegl1 \
    libgl1-mesa-glx

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists...
Building dependency tree...
Reading state information...
libegl-dev is already the newest version (1.4.0-1).
libegl1 is already the newest version (1.4.0-1).
libgles2 is already the newest version (1.4.0-1).
libglvnd0 is already the newest version (1.4.0-1).
libglew-dev is already the newest version (2.2.0-4).
libglfw3 is already the newest version (3.3.6-1).
libosmesa6-dev is already the newest version (23.2.1-1ubuntu3.1~22.04.4).
libgl1-mesa-glx is already the newest version (23.0.4-0ubuntu1~22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 125 not upgraded.


In [4]:
# Step 2 of 6 — PyTorch 2.2.0 (cu121)
# Must run BEFORE flash-attn: flash-attn compiles CUDA kernels against installed torch headers.
!pip install torch==2.2.0 torchvision==0.17.0 torchaudio==2.2.0 \
    --index-url https://download.pytorch.org/whl/cu121 -q

In [5]:
# Step 3 of 6 — MuJoCo + simulation stack
#
# Root cause of earlier failures — several pins predate Python 3.11/3.12:
#   mujoco 2.3.7   : no cp311/cp312 binary wheel; Python 3.12 removed
#                    distutils which mujoco's setup.py requires for source build.
#   numpy 1.22.4   : no cp311/cp312 binary wheel; source build needs Cython+Fortran.
#   opencv-python  : 4.6.0.66 has no cp311/cp312 binary wheel; source build is huge.
#
# Fix strategy:
#   mujoco  → conda-forge (pre-compiled binary for all Python versions)
#   numpy   → relaxed to >=1.24 which has cp312 wheels (LIBERO-compatible)
#   opencv  → relaxed to >=4.7 which has cp312 wheels (LIBERO-compatible)
#   gym     → --prefer-binary to avoid source build on Python 3.11+
import subprocess, sys
print(f"Python {sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}")

# numpy: must come before mujoco so conda doesn't downgrade it
!pip install "numpy>=1.24,<2" -q

# mujoco 2.3.7: conda-forge has pre-compiled binaries for all Python versions
# (pip source build requires distutils, removed in Python 3.12)
!conda install -c conda-forge -y -q mujoco=2.3.7 2>&1 | tail -5

# gym: 0.25.2 has no cp311/cp312 wheel; --prefer-binary uses nearest available
!pip install "gym>=0.21,<=0.26" --prefer-binary -q

# robosuite MUST be 1.4.0 — 1.5.x removed SingleArmEnv which LIBERO requires
# remaining deps are pure Python (no wheel issues)
!pip install \
    robosuite==1.4.0 \
    bddl==1.0.1 \
    easydict==1.9 \
    cloudpickle==2.1.0 \
    einops==0.4.1 \
    "imageio[ffmpeg]" -q

# opencv: 4.6.0.66 has no cp311/cp312 wheel; 4.7+ does
!pip install "opencv-python>=4.7,<5" --prefer-binary -q

# Verify mujoco importable (conda install can sometimes silently fail)
import importlib
if importlib.util.find_spec("mujoco") is None:
    raise RuntimeError("mujoco not importable after install — check conda output above")
print("✓ simulation stack installed")

  Preparing metadata (setup.py) ... done
  error: subprocess-exited-with-error
  
  × python setup.py bdist_wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for mujoco
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (mujoco)
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates fr

In [6]:
# Step 4 of 6 — LIBERO editable install
# LIBERO_PKG is defined in Cell 1. setup.py may be missing if Google Drive
# didn't sync the nested LIBERO git repo — auto-clone from GitHub in that case.
import os, subprocess, sys
from pathlib import Path

# Auto-clone LIBERO if setup.py not found on Drive
_libero_setup = Path(LIBERO_PKG) / 'setup.py'
if not _libero_setup.exists():
    _fallback = '/content/libero'
    print(f'⚠ {LIBERO_PKG}/setup.py missing (Google Drive may not sync nested git repos)')
    print(f'  Cloning LIBERO from GitHub → {_fallback} ...')
    _r = subprocess.run(
        ['git', 'clone', '--depth=1',
         'https://github.com/Lifelong-Robot-Learning/LIBERO.git', _fallback],
        capture_output=True, text=True,
    )
    if _r.returncode != 0:
        raise RuntimeError(f'LIBERO clone failed:\n{_r.stderr}')
    # Redirect paths to cloned location (affects Block B cells)
    LIBERO_PKG  = _fallback
    LIBERO_ROOT = f'{_fallback}/libero/libero'
    BDDL_FILE   = (
        f'{LIBERO_ROOT}/bddl_files/libero_spatial/'
        'pick_up_the_black_bowl_from_table_center_and_place_it_on_the_plate.bddl'
    )
    print(f'✓ LIBERO cloned. Paths updated → LIBERO_PKG={LIBERO_PKG}')
else:
    print(f'✓ LIBERO found at {LIBERO_PKG}')

# editable install — LIBERO package changes are live without reinstall
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-e', LIBERO_PKG, '-q'],
    capture_output=True, text=True,
)
if result.returncode != 0:
    print('STDERR:', result.stderr[-500:])
else:
    print('✓ LIBERO installed in editable mode from:', LIBERO_PKG)

(no stdout)
STDERR: ERROR: file:///content/drive/MyDrive/SoARM-Research/LIBERO does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.



In [7]:
# Step 5 of 6 — OpenVLA-OFT supporting packages + custom transformers fork
# The git fork (moojink/transformers-openvla-oft) MUST be installed LAST in this cell.
# Do NOT install transformers from PyPI — the fork replaces it entirely.
# The fork adds bidirectional attention for parallel decoding; PyPI version lacks this.
!pip install \
    timm==0.9.10 \
    tokenizers==0.19.1 \
    sentencepiece==0.1.99 \
    peft==0.11.1 \
    accelerate \
    huggingface_hub -q

# Install transformers fork LAST — pip resolver cannot downgrade to PyPI version this way.
# Commit SHA comment for reproducibility: installs from main branch of the fork repo.
# To pin a specific commit: git+https://github.com/moojink/transformers-openvla-oft.git@<SHA>
!pip install git+https://github.com/moojink/transformers-openvla-oft.git -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 6.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/2.6 MB 113.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
ERROR: Cannot install huggingface_hub>=1.2.0, timm==0.9.10 and tokenizers==0.19.1 because these package versions have conflicting dependencies.
ERROR: ResolutionImpossible: for help visit https://pip.pypa.io/en/latest/topics/dependency-resolution/#dealing-with-dependency-conflicts
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 112.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.6.0 requires transformer

In [ ]:
# Step 6 of 6 — flash-attn (must be last install)
# This cell compiles CUDA kernels — ~5-10 min on A100, 30-60+ min on T4. Do not interrupt.
# Must run AFTER torch is installed: flash-attn compiles against torch CUDA headers.
!pip install packaging ninja -q
!pip install "flash-attn==2.5.5" --no-build-isolation -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.7/180.7 kB 13.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 84.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


: 

---

## *** STOP — Restart runtime now ***

Go to: **Runtime > Restart session** (or press Ctrl+M .), then continue from BLOCK B below.

Do **not** run any cells below this point until after the runtime has restarted.

After restart, continue in **this notebook** — run the BLOCK B cells below.

---